# 🧠 Customer Feedback Sentiment Analysis — TF-IDF + Logistic Regression

**Goal:** Build a simple, interpretable machine learning pipeline that classifies customer reviews as **Positive**, **Negative**, or **Neutral**.

This notebook walks through the complete workflow:

1. Load & explore the dataset
2. Clean and preprocess the text
3. Visualize the data (class balance, review length, word clouds)
4. Vectorize text with **TF-IDF**
5. Train a **Logistic Regression** classifier
6. Evaluate the model (accuracy, classification report, confusion matrix)
7. Inspect which words drive each sentiment
8. Test the model on custom sentences
9. Save the trained model for reuse

> 📌 **Note on the dataset:** This notebook expects a CSV with two columns — `Comment` (the review text) and `Sentiment` (an integer label: `0` = negative, `1` = neutral, `2` = positive). If you're running this on Kaggle, attach your dataset via **Add Data** and update the `DATA_PATH` variable in the next section to point at it (Kaggle datasets are mounted under `/kaggle/input/`).


## 1. Import Libraries

In [ ]:
# Core data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text preprocessing helpers
import re
import string

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Model persistence
import joblib

# Word cloud (great for visual EDA on text data)
# If this isn't installed in your environment, run: !pip install wordcloud
from wordcloud import WordCloud

# Plot styling
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42


## 2. Load the Dataset

Update `DATA_PATH` below to match wherever your dataset lives.

- **On Kaggle:** after attaching your dataset, the path usually looks like `/kaggle/input/<dataset-name>/<file-name>.csv`. You can list available input files with the commented-out snippet below.
- **Locally:** point it at a CSV on disk, e.g. `"data/sentiment_data.csv"`.


In [ ]:
# Uncomment to explore available Kaggle input files:
# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

DATA_PATH = "/kaggle/input/your-dataset-folder/sentiment_data.csv"  # <-- update this

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")
df.head()


## 3. Initial Exploration

Before doing any cleaning, let's understand what we're working with — column types, missing values, and duplicates.


In [ ]:
df.info()


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate reviews (by Comment column):", df.duplicated(subset=["Comment"]).sum())


In [ ]:
df["Sentiment"].value_counts().sort_index()


## 4. Data Cleaning & Preprocessing

Steps applied:

1. Drop the stray `Unnamed: 0` index column if present (common artifact from exporting a CSV with `index=True`)
2. Rename `Comment` → `review_text` and `Sentiment` → `sentiment` for clarity
3. Drop rows with missing text or labels
4. Remove duplicate reviews
5. Map numeric sentiment codes (`0/1/2`) to human-readable labels (`negative/neutral/positive`)


In [ ]:
SENTIMENT_MAP = {0: "negative", 1: "neutral", 2: "positive"}

def clean_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    data = raw_df.copy()

    if "Unnamed: 0" in data.columns:
        data = data.drop(columns=["Unnamed: 0"])

    data = data.rename(columns={"Comment": "review_text", "Sentiment": "sentiment"})
    data = data.dropna(subset=["review_text", "sentiment"])
    data = data.drop_duplicates(subset=["review_text"])
    data["sentiment_label"] = data["sentiment"].map(SENTIMENT_MAP)

    return data

clean_df = clean_data(df)
print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning:  {len(clean_df):,}")
clean_df.head()


### Optional: light text normalization

Logistic Regression + TF-IDF works well even on raw text, but light normalization (lowercasing, stripping punctuation/URLs) often reduces noise and vocabulary size slightly. This step is optional — feel free to skip it and compare results.


In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)          # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)               # remove punctuation/special chars
    text = re.sub(r"\s+", " ", text).strip()               # collapse whitespace
    return text

clean_df["review_text_clean"] = clean_df["review_text"].apply(normalize_text)
clean_df[["review_text", "review_text_clean"]].head()


## 5. Exploratory Data Analysis (EDA)

### 5.1 Class Distribution

Is the dataset balanced across sentiment classes? This matters because a skewed dataset can bias the model toward the majority class.


In [ ]:
order = ["negative", "neutral", "positive"]
palette = {"negative": "#e74c3c", "neutral": "#95a5a6", "positive": "#2ecc71"}

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=clean_df, x="sentiment_label", order=order, palette=palette)
ax.set_title("Distribution of Sentiment Classes", fontsize=14, fontweight="bold")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Number of Reviews")

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()


In [ ]:
# Same information as percentages
class_pct = clean_df["sentiment_label"].value_counts(normalize=True).reindex(order) * 100
class_pct.plot(
    kind="pie",
    autopct="%1.1f%%",
    colors=[palette[c] for c in order],
    ylabel="",
    title="Sentiment Share (%)",
    figsize=(6, 6),
)
plt.tight_layout()
plt.show()


### 5.2 Review Length Analysis

Do longer reviews tend to skew toward a particular sentiment? Let's look at word counts per review.


In [ ]:
clean_df["word_count"] = clean_df["review_text_clean"].apply(lambda x: len(x.split()))

plt.figure(figsize=(9, 5))
sns.histplot(data=clean_df, x="word_count", hue="sentiment_label", hue_order=order,
             palette=palette, bins=40, kde=True, element="step")
plt.title("Review Length Distribution by Sentiment", fontsize=14, fontweight="bold")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=clean_df, x="sentiment_label", y="word_count", order=order, palette=palette)
plt.title("Word Count Spread by Sentiment", fontsize=14, fontweight="bold")
plt.xlabel("Sentiment")
plt.ylabel("Word Count")
plt.tight_layout()
plt.show()


### 5.3 Word Clouds

A quick visual gut-check of the most frequent words in each sentiment class.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, label in zip(axes, order):
    text_blob = " ".join(clean_df.loc[clean_df["sentiment_label"] == label, "review_text_clean"])
    wc = WordCloud(width=600, height=400, background_color="white",
                   colormap="viridis", max_words=100).generate(text_blob)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"{label.capitalize()} Reviews", fontsize=14, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 6. Train / Test Split

We use an 80/20 stratified split so the sentiment distribution is preserved in both sets.


In [ ]:
X = clean_df["review_text_clean"]
y = clean_df["sentiment_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples:  {len(X_test):,}")


## 7. Feature Extraction — TF-IDF

**TF-IDF (Term Frequency – Inverse Document Frequency)** converts text into numeric vectors by weighting words based on how often they appear in a document versus across the whole corpus. Words that are common everywhere (like "the", "and") get down-weighted, while distinctive words get more weight.

We cap the vocabulary at 5,000 features to keep the model fast and reduce overfitting risk.


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape (train): {X_train_vec.shape}")
print(f"TF-IDF matrix shape (test):  {X_test_vec.shape}")


## 8. Model Training — Logistic Regression

Logistic Regression is a strong, fast, and interpretable baseline for text classification — a great starting point before reaching for heavier models (SVM, gradient boosting, or transformer-based models like BERT).


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_vec, y_train)

print("Model training complete ✅")


## 9. Model Evaluation

### 9.1 Accuracy & Classification Report


In [ ]:
y_pred = model.predict(X_test_vec)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=order))


### 9.2 Confusion Matrix

The confusion matrix shows exactly which sentiment classes the model confuses with each other.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred, labels=order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=order)
disp.plot(ax=ax, cmap="Blues", colorbar=True, values_format="d")
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


### 9.3 Precision / Recall / F1 by Class (visual)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, labels=order)
metrics_df = pd.DataFrame({"precision": precision, "recall": recall, "f1-score": f1}, index=order)

metrics_df.plot(kind="bar", figsize=(9, 5), color=["#3498db", "#f1c40f", "#9b59b6"])
plt.title("Precision, Recall & F1-score by Sentiment Class", fontsize=14, fontweight="bold")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

metrics_df


## 10. Model Interpretability — Most Influential Words

Because Logistic Regression is a linear model, its coefficients tell us directly which words push a prediction toward each sentiment class. Let's extract the top words for each class.


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, class_label in zip(axes, model.classes_):
    class_idx = list(model.classes_).index(class_label)
    coefs = model.coef_[class_idx]
    top_idx = np.argsort(coefs)[-15:]  # top 15 positive contributors for this class

    ax.barh(feature_names[top_idx], coefs[top_idx], color=palette.get(class_label, "#3498db"))
    ax.set_title(f"Top words → '{class_label}'", fontsize=13, fontweight="bold")
    ax.set_xlabel("Coefficient weight")

plt.tight_layout()
plt.show()


## 11. Testing on Custom Sentences

Let's sanity-check the model on a handful of hand-written examples that weren't part of the dataset.


In [ ]:
def predict_sentiment(text: str) -> str:
    cleaned = normalize_text(text)
    vec = vectorizer.transform([cleaned])
    return model.predict(vec)[0]

sample_reviews = [
    "This product is amazing, I love it!",
    "Terrible experience, would not recommend.",
    "It's okay, nothing special.",
    "The product works as expected.",
    "Average quality, does the job.",
    "I have no strong opinion about this.",
    "Received the item on time, standard packaging.",
]

for review in sample_reviews:
    print(f"{review!r:60s} -> {predict_sentiment(review)}")


## 12. Saving the Model

We persist both the trained classifier and the fitted TF-IDF vectorizer with `joblib`, so they can be reloaded later for inference without retraining.


In [ ]:
joblib.dump(model, "logistic_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Saved: logistic_model.pkl, tfidf_vectorizer.pkl")


## 13. Conclusion & Next Steps

**Summary:** We built an end-to-end sentiment classification pipeline — cleaning raw review text, exploring class balance and review length, converting text to numeric features with TF-IDF, and training a Logistic Regression classifier that reached a solid baseline accuracy.

**Ideas to extend this notebook:**
- Try other classifiers (Linear SVM, Naive Bayes, Random Forest, XGBoost) and compare metrics
- Use `GridSearchCV` to tune `max_features`, `ngram_range`, and `C` (regularization strength)
- Handle class imbalance with `class_weight="balanced"` or oversampling (SMOTE)
- Swap TF-IDF for word embeddings (Word2Vec, GloVe) or a transformer model (e.g. DistilBERT) for richer representations
- Add error analysis: manually inspect misclassified reviews to spot patterns

If you found this notebook useful, an upvote is always appreciated! 🙌
